<a href="https://colab.research.google.com/github/Daloer-LIFO/AI-project-for-LinkedIn-Post-creator/blob/main/Module_24_Assignment_rajkahini_rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Module 24 Assignment-Knowledge Base Chatbot with Vector DB**  
**BACKGROUND**


To build a Knowledge Base Chatbot that answers questions about one Bangla book using a RAG (Retrieval-Augmented Generation) pipeline.
Core Rules:

•	Every answer must come only from the selected book.

•	Every answer must show where the information came from by citing the relevant chapter/section.

•	If the answer is not available in the book, the chatbot must clearly say so instead of making up an answer.

Here, the assigned book is রাজকাহিনী by অবনীন্দ্রনাথ ঠাকুর, from Bengali Wikisource. The Wikisource main page identifies the work as রাজকাহিনী, by অবনীন্দ্রনাথ ঠাকুর, published in 1914, and its contents are distributed across chapter/subpages.

# 1. Book and Project Being Selected

The following book is being used:

Book: রাজকাহিনী  
Author: অবনীন্দ্রনাথ ঠাকুর  
Source: Bengali Wikisource  
URL: রাজকাহিনী — Bengali Wikisource  
RAG type: Retrieval-Augmented Generation  
Vector DB: FAISS  
Embedding: BAAI/bge-m3  
LLM: Gemini through LangChain  
Interface: Gradio  

*BAAI/bge-m3 is suitable here because it is a multilingual embedding model and supports long text; its model documentation describes multilingual retrieval capabilities and an 8192-token sequence length.*


In [2]:
# 2. Required Packages Being Installed

!pip install -qU \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-google-genai \
    faiss-cpu \
    sentence-transformers \
    beautifulsoup4 \
    requests \
    gradio \
    tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8

In [3]:
# 3. Required Libraries Being Imported
import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, unquote

import gradio as gr

from sentence_transformers import SentenceTransformer

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

/tmp/ipykernel_2726/720422711.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
# 4. API Key Being Configured/
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY was not found in Colab Secrets.")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("GOOGLE_API_KEY has been loaded successfully.")

In [ ]:
# 5. Book Source Being Defined
BOOK_NAME = "রাজকাহিনী"
AUTHOR = "অবনীন্দ্রনাথ ঠাকুর"

BASE_URL = "https://bn.wikisource.org"
BOOK_URL = "https://bn.wikisource.org/wiki/রাজকাহিনী"

print("Book:", BOOK_NAME)
print("Author:", AUTHOR)
print("Source:", BOOK_URL)

'''
The Wikisource book contains separate pages such as গোহ, বাপ্পাদিত্য, and পদ্মিনী,
demonstrating why the assignment requires crawling the relevant subpages
rather than ingesting only the main page.
'''

In [1]:
# 6. Chapter Links Being Crawled
headers = {
    "User-Agent": "Mozilla/5.0 (Knowledge Base Assignment)"
}

response = requests.get(
    BOOK_URL,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

chapter_urls = set()

for link in soup.find_all("a", href=True):
    href = link["href"]

    if href.startswith("/wiki/রাজকাহিনী/"):
        full_url = urljoin(BASE_URL, href)

        # Remove fragment
        full_url = full_url.split("#")[0]

        chapter_urls.add(full_url)

chapter_urls = sorted(chapter_urls)

print("Number of discovered chapter/subpages:", len(chapter_urls))

for url in chapter_urls:
    print(url)

'''
The assignment explicitly requires crawling all relevant chapter/subpage URLs
rather than ingesting only the first page.

This approach allows the notebook to discover the chapter pages automatically
instead of hard-coding only গোহ, বাপ্পাদিত্য, etc.
'''

In [ ]:
# 7. Chapter Pages Being Verified
for i, url in enumerate(chapter_urls, start=1):
    print(f"{i}. {unquote(url)}")

'''
OUTPUTS:
https://bn.wikisource.org/wiki/রাজকাহিনী/গোহ
https://bn.wikisource.org/wiki/রাজকাহিনী/বাপ্পাদিত্য
https://bn.wikisource.org/wiki/রাজকাহিনী/পদ্মিনী

The actual pages contain the book text; for example,
the Wikisource page for গোহ identifies it as part of রাজকাহিনী
and gives its page range.
'''

In [ ]:
# 8. Book Text Being Extracted
def clean_wikisource_page(url):
    """
    Downloads a Wikisource chapter page and extracts the main text.
    """

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Main article area
    content = soup.select_one("#mw-content-text")

    if content is None:
        return ""

    # Remove unwanted elements
    for element in content.select(
        "table, style, script, .mw-editsection, "
        ".navbox, .metadata, .noprint, .thumb"
    ):
        element.decompose()

    text = content.get_text("\n")

    # Normalize whitespace
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [ ]:
'''
9. Chapter Metadata Being Preserved

The assignment requires useful metadata including:

Book name
Chapter name
Section name
Source URL
'''

documents = []

for url in chapter_urls:

    chapter_name = unquote(
        urlparse(url).path.split("/")[-1]
    )

    text = clean_wikisource_page(url)

    if len(text) < 100:
        print("Skipped:", chapter_name)
        continue

    metadata = {
        "book": BOOK_NAME,
        "author": AUTHOR,
        "chapter": chapter_name,
        "section": chapter_name,
        "source_url": url
    }

    documents.append(
        Document(
            page_content=text,
            metadata=metadata
        )
    )

    print(
        f"Loaded: {chapter_name} | "
        f"Characters: {len(text)}"
    )

print("\nTotal documents:", len(documents))

In [ ]:
# 10. Extracted Book Data Being Checked

for doc in documents[:3]:

    print("=" * 80)
    print("Chapter:", doc.metadata["chapter"])
    print("URL:", doc.metadata["source_url"])
    print("-" * 80)
    print(doc.page_content[:1500])
    print()

'''
This check is important because the chatbot must answer only from the selected book.
The assignment explicitly prohibits fabricated answers.
'''

In [ ]:
'''
11. Text Cleaning Being Improved

Some Wikisource pages contain navigation text or formatting artifacts.
A small normalization step is therefore being applied.
'''

def preprocess_text(text):

    # Remove repeated whitespace
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove common Wikisource artifacts
    unwanted_patterns = [
        r"সম্পাদনা",
        r"বিষয়শ্রেণীসমূহ:",
        r"ভাষা যোগ করুন",
        r"আন্তঃউইকি সংযোগ দিন"
    ]

    for pattern in unwanted_patterns:
        text = re.sub(pattern, "", text)

    return text.strip()


for doc in documents:
    doc.page_content = preprocess_text(doc.page_content)

print("Preprocessing completed.")

In [ ]:
# 12. Text Chunks Being Created

'''
For this project, the following chunking configuration is being used:
Chunk size   = 800 characters
Chunk overlap = 150 characters

The overlap is being used so that information occurring near chunk boundaries is less likely to be lost.

The assignment explicitly requires the chunk size, overlap, and preprocessing method to be documented.
'''
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        "।",
        "৷",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Generated chunks:", len(chunks))

In [ ]:
# 13. Chunk Metadata Being Verified
for i, chunk in enumerate(chunks[:5]):

    print("=" * 80)
    print("Chunk:", i + 1)
    print("Book:", chunk.metadata["book"])
    print("Chapter:", chunk.metadata["chapter"])
    print("Source:", chunk.metadata["source_url"])
    print("Text:")
    print(chunk.page_content[:500])

In [ ]:
'''
14. Multilingual Embeddings Being Loaded

BAAI/bge-m3 is being selected because the assignment requires a multilingual embedding model
capable of handling Bengali rather than an English-only model.

The model card shows that BGE-M3 can be used with Sentence Transformers.
'''

embedding_model_name = "BAAI/bge-m3"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded:", embedding_model_name)

In [ ]:
'''
15. FAISS Vector Database Being Built

The assignment allows either FAISS or Chroma.
FAISS is being used here because it is straightforward to create and save in Colab.
'''

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created successfully.")

In [1]:
# 16. Vector Database Being Saved
VECTOR_DB_PATH = "/content/rajkahini_faiss"

vector_db.save_local(VECTOR_DB_PATH)

print("FAISS database saved to:", VECTOR_DB_PATH)

In [ ]:
# 17. Retriever Being Configured

# The assignment requires a LangChain retriever + LLM.

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("Retriever configured.")

In [ ]:
# 18. Retrieval Being Tested

'''
This is an important demonstration for assignment because
it proves that the question is being converted into a retrieval request
and relevant book passages are being returned.
'''
test_query = "বাপ্পাদিত্য কে ছিলেন?"

retrieved_docs = retriever.invoke(test_query)

for i, doc in enumerate(retrieved_docs, start=1):

    print("=" * 80)
    print("Retrieved passage:", i)
    print("Chapter:", doc.metadata["chapter"])
    print("Source:", doc.metadata["source_url"])
    print("-" * 80)
    print(doc.page_content[:800])

In [ ]:
# 19. Gemini LLM Being Configured
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Gemini LLM initialized.")

'''
temperature=0 is being used because the assignment emphasizes factual answers grounded
in the book and avoidance of hallucination.

In [ ]:
'''
20. RAG Prompt Being Designed

The prompt is explicitly telling Gemini:

1.Use only the retrieved book context.
2.Do not use outside knowledge.
3.Do not invent information.
4.Say that the information is unavailable if it is not in the context.
5.Include chapter/source citation.
'''

prompt = ChatPromptTemplate.from_template("""
তুমি "রাজকাহিনী" বইয়ের একটি Knowledge Base Chatbot হিসেবে কাজ করছ।

কঠোর নিয়ম:

1. শুধুমাত্র নিচে দেওয়া BOOK CONTEXT ব্যবহার করে উত্তর দাও।
2. তোমার নিজের সাধারণ জ্ঞান ব্যবহার করবে না।
3. BOOK CONTEXT-এ উত্তর পাওয়া না গেলে স্পষ্টভাবে বলবে:
   "এই তথ্যটি রাজকাহিনী বইয়ের প্রদত্ত অংশে পাওয়া যায়নি।"
4. কোনো তথ্য অনুমান বা বানিয়ে লিখবে না।
5. উত্তর বাংলায় প্রদান করবে।
6. উত্তরের শেষে Chapter এবং Source URL উল্লেখ করবে।
7. Context-এ একাধিক chapter থাকলে সবচেয়ে প্রাসঙ্গিক chapter উল্লেখ করবে।

BOOK CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
""")


In [ ]:
# 21. Retrieved Context Being Formatted
def format_docs(docs):

    formatted = []

    for doc in docs:

        formatted.append(
            f"""
অধ্যায়: {doc.metadata.get('chapter', 'অজানা')}
বই: {doc.metadata.get('book', BOOK_NAME)}
Source URL: {doc.metadata.get('source_url', '')}

বইয়ের অংশ:
{doc.page_content}
"""
        )

    return "\n\n".join(formatted)

In [ ]:
# 22. RAG Pipeline Being Connected

def answer_question(question):

    # Step 1: Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Step 2: Format retrieved context
    context = format_docs(retrieved_docs)

    # Step 3: Create prompt
    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    # Step 4: Generate answer
    response = llm.invoke(messages)

    return response.content

The resulting flow is:

User Question  
      ↓  
Query Embedding  
      ↓  
FAISS Search  
      ↓  
Relevant Book Chunks  
      ↓  
Context  
      ↓  
Gemini  
      ↓  
Answer + Citation

In [ ]:
# 23. RAG Pipeline Being Tested
# First Question Being Asked
question = "বাপ্পাদিত্য কে ছিলেন?"

answer = answer_question(question)

print(answer)

In [ ]:
# 24. No-Answer Protection Being Tested

'''This is particularly important because the assignment specifically requires
at least one question whose answer is not present in the book.
'''

# Out-of-Book Question Being Tested
question = "বাংলাদেশের রাজধানী কী?"

answer = answer_question(question)

print(answer)

In [ ]:
# 25. Source Citation Being Improved

# For a stronger demonstration, the citations can be generated outside the LLM so they cannot be accidentally fabricated.

def answer_question_with_sources(question):

    retrieved_docs = retriever.invoke(question)

    context = format_docs(retrieved_docs)

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    unique_sources = []

    for doc in retrieved_docs:

        source = {
            "chapter": doc.metadata.get("chapter"),
            "url": doc.metadata.get("source_url")
        }

        if source not in unique_sources:
            unique_sources.append(source)

    return response.content, unique_sources

In [ ]:
# 26. Citation Information Being Displayed
question = "গোহের সঙ্গে ভীলদের সম্পর্ক কী ছিল?"

answer, sources = answer_question_with_sources(question)

print("ANSWER")
print("=" * 80)
print(answer)

print("\nSOURCES")
print("=" * 80)

for source in sources:
    print("Chapter:", source["chapter"])
    print("URL:", source["url"])
    print()

In [ ]:
# 27. Gradio Chat Interface Being Created
'''
The assignment allows Gradio as the frontend and requires
the user to be able to enter a question, receive an answer,
and see the source/chapter citation.
'''

def chatbot(question):

    if not question or not question.strip():
        return "দয়া করে একটি প্রশ্ন লিখুন।"

    answer, sources = answer_question_with_sources(question)

    citation_text = "\n\n### 📚 Source / Citation\n"

    for source in sources:
        citation_text += (
            f"- অধ্যায়: {source['chapter']}\n"
            f"- Source: {source['url']}\n"
        )

    return answer + citation_text

In [ ]:
# 28. User Interface Being Launched
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="আপনার প্রশ্ন",
        placeholder="রাজকাহিনী সম্পর্কে একটি প্রশ্ন করুন..."
    ),
    outputs=gr.Markdown(
        label="উত্তর"
    ),
    title="📚 রাজকাহিনী — Knowledge Base Chatbot",
    description=(
        "এই chatbot শুধুমাত্র অবনীন্দ্রনাথ ঠাকুরের "
        "রাজকাহিনী বইয়ের তথ্য ব্যবহার করে উত্তর প্রদান করে।"
    )
)

demo.launch(share=True)

# 29. Ten Test Questions Being Prepared
The assignment requires 10 test questions, including at least one no-answer case.
| No. | Test Question                                               | Purpose                |
| --: | ----------------------------------------------------------- | ---------------------- |
|   1 | শিলাদিত্য কে ছিলেন?                                         | Character retrieval    |
|   2 | গোহের কাহিনিতে পুষ্পবতীর ভূমিকা কী?                         | Character relationship |
|   3 | গোহ কীভাবে ভীলরাজের সিংহাসনে বসেছিলেন?                      | Event retrieval        |
|   4 | বাপ্পাদিত্য কীভাবে যুদ্ধে নেতৃত্ব দেওয়ার দায়িত্ব পেয়েছিলেন? | Event retrieval        |
|   5 | বাপ্পাদিত্য গল্পে রাজপুতদের অবস্থান কী ছিল?                 | Context retrieval      |
|   6 | পদ্মিনীর কাহিনির সঙ্গে চিতোরের কী সম্পর্ক রয়েছে?            | Chapter retrieval      |
|   7 | রাজকাহিনীতে কোন কোন রাজা বা রাজপুত চরিত্রের উল্লেখ রয়েছে?   | Entity retrieval       |
|   8 | রাজকাহিনীর মূল বিষয়বস্তু কী?                                | Summary retrieval      |
|   9 | পদ্মিনী অধ্যায়ে বর্ণিত গুরুত্বপূর্ণ ঘটনাটি কী?              | Chapter understanding  |
|  10 | বাংলাদেশের বর্তমান রাষ্ট্রপতির নাম কী?                      | **No-answer test**     |


In [4]:
# 30. Ten Test Questions Being Executed
test_questions = [
    "শিলাদিত্য কে ছিলেন?",
    "গোহের কাহিনিতে পুষ্পবতীর ভূমিকা কী?",
    "গোহ কীভাবে ভীলরাজের সিংহাসনে বসেছিলেন?",
    "বাপ্পাদিত্য কীভাবে যুদ্ধে নেতৃত্ব দেওয়ার দায়িত্ব পেয়েছিলেন?",
    "বাপ্পাদিত্য গল্পে রাজপুতদের অবস্থান কী ছিল?",
    "পদ্মিনীর কাহিনির সঙ্গে চিতোরের কী সম্পর্ক রয়েছে?",
    "রাজকাহিনীতে কোন কোন রাজা বা রাজপুত চরিত্রের উল্লেখ রয়েছে?",
    "রাজকাহিনীর মূল বিষয়বস্তু কী?",
    "পদ্মিনী অধ্যায়ে বর্ণিত গুরুত্বপূর্ণ ঘটনাটি কী?",
    "বাংলাদেশের বর্তমান রাষ্ট্রপতির নাম কী?"
]

In [ ]:
results = []

for i, question in enumerate(test_questions, start=1):

    print("=" * 100)
    print(f"QUESTION {i}")
    print(question)

    answer = answer_question(question)

    print("\nANSWER")
    print(answer)

    results.append({
        "question": question,
        "answer": answer
    })


In [ ]:
# 31. Test Results Being Saved
with open(
    "/content/test_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Test results saved.")

# 32. Bonus Retrieval Comparison Being Added



For simplicity, two chunking strategies are being compared:

Strategy A → 500 characters / 100 overlap  
Strategy B → 800 characters / 150 overlap

In [ ]:
# 33. Alternative Chunking Strategy Being Created
splitter_a = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        "।",
        "৷",
        " ",
        ""
    ]
)

chunks_a = splitter_a.split_documents(documents)

print("Strategy A chunks:", len(chunks_a))

In [ ]:
# 34. Second Chunking Strategy Being Created
splitter_b = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        "।",
        "৷",
        " ",
        ""
    ]
)

chunks_b = splitter_b.split_documents(documents)

print("Strategy B chunks:", len(chunks_b))

In [ ]:
# 35. Two Vector Databases Being Created
vector_a = FAISS.from_documents(
    chunks_a,
    embeddings
)

vector_b = FAISS.from_documents(
    chunks_b,
    embeddings
)

retriever_a = vector_a.as_retriever(
    search_kwargs={"k": 5}
)

retriever_b = vector_b.as_retriever(
    search_kwargs={"k": 5}
)

print("Both retrieval systems have been created.")

In [ ]:
# 36. Hit-Rate Evaluation Being Performed
evaluation_set = [
    {
        "question": "গোহের কাহিনিতে পুষ্পবতীর ভূমিকা কী?",
        "expected_chapter": "গোহ"
    },
    {
        "question": "বাপ্পাদিত্য কীভাবে যুদ্ধে নেতৃত্ব দেওয়ার দায়িত্ব পেয়েছিলেন?",
        "expected_chapter": "বাপ্পাদিত্য"
    },
    {
        "question": "পদ্মিনীর কাহিনির সঙ্গে চিতোরের কী সম্পর্ক রয়েছে?",
        "expected_chapter": "পদ্মিনী"
    }
]

In [ ]:
# 37. Retrieval Hit Rate Being Calculated
def calculate_hit_rate(retriever, evaluation_data):

    hits = 0

    for item in evaluation_data:

        docs = retriever.invoke(item["question"])

        retrieved_chapters = [
            doc.metadata.get("chapter", "")
            for doc in docs
        ]

        expected = item["expected_chapter"]

        if expected in retrieved_chapters:
            hits += 1

    return hits / len(evaluation_data)

In [ ]:
# 38. Two Strategies Being Compared
hit_rate_a = calculate_hit_rate(
    retriever_a,
    evaluation_set
)

hit_rate_b = calculate_hit_rate(
    retriever_b,
    evaluation_set
)

print(f"Strategy A Hit Rate: {hit_rate_a:.2%}")
print(f"Strategy B Hit Rate: {hit_rate_b:.2%}")

# 39. RAG Architecture:

                 ┌──────────────────────┐
                 │ Bengali Wikisource   │
                 │     রাজকাহিনী       │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │   Web Crawling       │
                 │ Chapter/Subpages     │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Cleaning &           │
                 │ Preprocessing         │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Text Chunking        │
                 │ 800 / 150            │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ BGE-M3 Embeddings    │
                 │ Multilingual/Bengali │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ FAISS Vector DB      │
                 └──────────┬───────────┘
                            │
                  User Question
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Query Embedding      │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Similarity Search    │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Relevant Context     │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Gemini LLM           │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Answer + Citation    │
                 └──────────────────────┘